# BIRD-SQL Evaluation — Gap 3 (chuẩn hoá evidence): EX & VES

Chạy `evaluation.py` (EX) và `evaluation_ves.py` (VES) gốc, **không sửa gì**, trên 8 file đã ghép (`*_v2`):
mỗi model có 1 file **đối chứng** (`kg_ctrl_v2`, prompt gốc) và 1 file **chuẩn hoá** (`kg_norm_v2`, `--normalize_evidence`).
Mỗi file đủ 1534 câu: 182 câu có evidence bị đổi được thay bằng kết quả chạy mới, 1352 câu còn lại lấy từ baseline.

**Chuẩn bị:**
1. Upload `bird-gap3-eval.zip` (thư mục gốc repo) thành Kaggle dataset, ví dụ `bird-gap3-eval`.
2. *Add Input*: dataset đó **và** dataset BIRD (`bird-sql`, chứa `llm/src/evaluation*.py` và `llm/data/dev_gold.sql`).
3. Không cần GPU (chỉ chạy SQLite). Chạy *Run All*.

⚠️ Ollama: 1352 câu nền đang lấy từ file full cũ (prompt trước khi sửa bug) → số toàn tập của Ollama là **tạm thời**, chạy lại sau khi có baseline full mới. GPT-3.5 thì nền là `openai_output` (đúng prompt hiện tại).

In [ ]:
import os

BIRD_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dev.json' in files and os.path.basename(root) == 'data':
        BIRD_ROOT = os.path.dirname(root)  # thư mục cha của "data/"
        break

assert BIRD_ROOT is not None, "Không tìm thấy bird/data/dev.json trong /kaggle/input — kiểm tra lại dataset đã attach chưa"
print('BIRD_ROOT =', BIRD_ROOT)
print('dev_databases exists:', os.path.exists(f'{BIRD_ROOT}/data/dev_databases'))

import shutil
LOCAL_DB_ROOT = '/kaggle/working/dev_databases'
if not os.path.exists(LOCAL_DB_ROOT):
    shutil.copytree(f'{BIRD_ROOT}/data/dev_databases', LOCAL_DB_ROOT)
print('DBs copied:', len(os.listdir(LOCAL_DB_ROOT)))



In [ ]:
import shutil, os

LOCAL_SRC = '/kaggle/working/src'
os.makedirs(LOCAL_SRC, exist_ok=True)
shutil.copy(f'{BIRD_ROOT}/llm/src/evaluation.py', LOCAL_SRC)
shutil.copy(f'{BIRD_ROOT}/llm/src/evaluation_ves.py', LOCAL_SRC)

# Kiểm tra xác nhận file đã có --iterate_num sẵn (không cần patch nữa)
has_fix = 'iterate_num' in open(f'{LOCAL_SRC}/evaluation_ves.py').read()
print('evaluation_ves.py đã có --iterate_num:', has_fix)


In [ ]:
# Tìm thư mục chứa các file *_v2 (dataset bird-gap3-eval)
import glob

hits = glob.glob('/kaggle/input/**/deepseek_kg_ctrl_v2/predict_dev.json', recursive=True)
assert hits, f'Không thấy *_v2 — attach dataset bird-gap3-eval chưa? Đang attach: {os.listdir("/kaggle/input")}'
RESULT_ROOT = os.path.dirname(os.path.dirname(hits[0]))
print('RESULT_ROOT =', RESULT_ROOT)
print(sorted(d for d in os.listdir(RESULT_ROOT) if d.endswith('_v2')))

In [ ]:
import subprocess, sys, re
import pandas as pd

# model hiển thị -> (thư mục đối chứng, thư mục chuẩn hoá)
MODEL_FOLDERS = {
    "GPT-3.5-Turbo":     ("turbo_kg_ctrl_v2",    "turbo_kg_norm_v2"),
    "DeepSeek-Coder-V2": ("deepseek_kg_ctrl_v2", "deepseek_kg_norm_v2"),
    "Gemma3":            ("gemma3_kg_ctrl_v2",   "gemma3_kg_norm_v2"),
    "Qwen2.5-Coder":     ("qwen25_kg_ctrl_v2",   "qwen25_kg_norm_v2"),
}

GT_DIR = f'{BIRD_ROOT}/llm/data'
DIFF_JSON = f'{BIRD_ROOT}/data/dev.json'

# Kiểm tra đủ 8 folder trước khi chạy
for name, (ctrl_dir, norm_dir) in MODEL_FOLDERS.items():
    for label, d in [("ctrl", ctrl_dir), ("norm", norm_dir)]:
        p = f'{RESULT_ROOT}/{d}/predict_dev.json'
        print(f"{name:20} {label:5} -> {'OK' if os.path.exists(p) else 'MISSING'}  {p}")

def run_eval_script(script_name, predicted_dir_name, extra_args=None):
    cmd = [
        sys.executable, f'{LOCAL_SRC}/{script_name}',
        '--db_root_path', LOCAL_DB_ROOT + '/',
        '--predicted_sql_path', f'{RESULT_ROOT}/{predicted_dir_name}/',
        '--ground_truth_path', GT_DIR + '/',
        '--data_mode', 'dev',
        '--num_cpus', '4',
        '--meta_time_out', '30.0',
        '--mode_gt', 'gt',
        '--mode_predict', 'gpt',
        '--diff_json_path', DIFF_JSON,
    ]
    if extra_args:
        cmd += extra_args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'[{predicted_dir_name}/{script_name}] STDERR:', result.stderr[-2000:])
    return result.stdout

SCORE_RE = re.compile(r"(accuracy|ves)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)")

def parse_score(stdout):
    m = SCORE_RE.search(stdout)
    if not m:
        return None
    _, s, mo, c, t = m.groups()
    return [float(s), float(mo), float(c), float(t)]

In [ ]:
# ex_scores[model_name][ctrl/norm] = [simple, moderate, challenging, total]
ex_scores = {}
ves_scores = {}

for name, (ctrl_dir, norm_dir) in MODEL_FOLDERS.items():
    ex_scores[name] = {}
    ves_scores[name] = {}
    for label, d in [("ctrl", ctrl_dir), ("norm", norm_dir)]:
        print(f"\n===== {name} ({label}) : EX =====")
        ex_out = run_eval_script('evaluation.py', d)
        print(ex_out[-500:])
        ex_scores[name][label] = parse_score(ex_out)

        print(f"\n===== {name} ({label}) : VES (iterate_num=5) =====")
        ves_out = run_eval_script('evaluation_ves.py', d, extra_args=['--iterate_num', '5'])
        print(ves_out[-500:])
        ves_scores[name][label] = parse_score(ves_out)

In [ ]:
# Bảng tổng hợp — chỉ lấy cột "total" (index 3) của [simple, moderate, challenging, total]
def total(score):
    return score[3] if score is not None else None

rows = {}
for name in MODEL_FOLDERS:
    ex_c, ex_n = total(ex_scores[name]["ctrl"]), total(ex_scores[name]["norm"])
    ves_c, ves_n = total(ves_scores[name]["ctrl"]), total(ves_scores[name]["norm"])
    rows[name] = {
        "EX_ctrl": ex_c, "EX_norm": ex_n,
        "EX_delta": round(ex_n - ex_c, 2) if None not in (ex_c, ex_n) else None,
        "VES_ctrl": ves_c, "VES_norm": ves_n,
        "VES_delta": round(ves_n - ves_c, 2) if None not in (ves_c, ves_n) else None,
    }

summary = pd.DataFrame(rows).T
summary.to_csv('/kaggle/working/summary_gap3_ex_ves.csv')
summary